### Importing Model

In [32]:
import message_ix
import ixmp as ix
from ixmp import Scenario
from message_ix import make_df
from pre_process.prov_parameters import generator,remove_canada_data, remove_technology_data
import pandas as pd
from pathlib import Path
from pre_process.update_calibration import model_calibrate
from pre_process.update_units import unit_correction
from pre_process.cleanup_rel_tec import cleanup_rel_tec
from message_ix.report import Reporter

import numpy as np

## Initialize scenario

In [3]:
 # Stand-alone model
modelName = 'MESSAGEix_CA_provincial'      # A MESSAGE model name
scenarioName = 'baseline'       # A MESSAGE model scenario name
ssp = 'SSP2'

mp = ix.Platform(name = 'p3')

# Creating a new MESSAGE stand-alone model
msg = message_ix.Scenario(mp, modelName, scenarioName,
                            version='new', annotation='')

2024-04-12 21:57:02,351  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2024-04-12 21:57:02,358  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp' (user: ixmp)...


In [4]:
# Path to data folder
DATA = Path().cwd().parent / 'data'
TEST = Path().cwd().parent / 'test'

In [5]:
# Read in data from Canadian scenario
msg.read_excel(DATA / 'data_CA_provinicial_v3.xlsx',
               add_units=True, 
               commit_steps=False,
               init_items = True)

In [6]:
msg.check_out()

### Adding Import & Export Data

In [7]:
df = pd.read_excel(DATA /'provincial_sources/historical_import_export.xlsx')

# Select columns up to 'unit'
df = df[['node_loc', 'technology', 'year_vtg', 'value', 'unit']].rename(columns={'year_vtg': 'year_act'})

# Add 'mode' column with all values set to 'M1'
df['mode'] = 'M1'
df['unit'] ='GWa'
# Add 'time' column with values from 'year_vtg'
df['time'] = 'year'
df['node_loc'] = df['node_loc'].str.replace(' ', '')['node_loc'] = df['node_loc'].str.replace(' ', '')
df['value'] = df['value'].fillna(0)


In [8]:
msg.add_par('historical_activity', df)

In [9]:
import pandas as pd

# Sample data to replicate across Canadian provinces
data = [
    ('R11_NAM', 'elec_t_d', 1990, 'M1', 'year', 346.902),
    ('R11_NAM', 'elec_t_d', 1995, 'M1', 'year', 414.603),
    ('R11_NAM', 'elec_t_d', 2000, 'M1', 'year', 455.097),
    ('R11_NAM', 'elec_t_d', 2005, 'M1', 'year', 482.681),
    ('R11_NAM', 'elec_t_d', 2010, 'M1', 'year', 484.257),
    ('R11_NAM', 'elec_t_d', 2015, 'M1', 'year', 485.843),
    ('R11_NAM', 'elec_t_d', 2020, 'M1', 'year', 521.053)
]

# Canadian provinces and territories with no spaces in names
provinces = [
    'Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 
    'NewfoundlandandLabrador', 'NovaScotia', 'Ontario', 
    'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 
    'NorthwestTerritories', 'Nunavut', 'Yukon'
]

# Create a DataFrame for the sample data
df = pd.DataFrame(data, columns=['node_loc', 'technology', 'year_act', 'mode', 'time', 'value'])

# List to store DataFrames for each province
province_dfs = []

# Replicate the data for each Canadian province
for province in provinces:
    temp_df = df.copy()
    temp_df['node_loc'] = province
    province_dfs.append(temp_df)

# Concatenate all provincial DataFrames into one
final_df = pd.concat(province_dfs, ignore_index=True)

msg.add_par('historical_activity', final_df)

In [10]:
# msg.check_out()
df = msg.par('initial_activity_lo')
msg.remove_par('initial_activity_lo', df)
filtered_df = df[~df['technology'].isin(['elec_imp','elec_exp'])]
msg.add_par('initial_activity_lo',filtered_df) 
# msg.check_out()
df = msg.par('initial_activity_up')
msg.remove_par('initial_activity_up', df)
filtered_df = df[~df['technology'].isin(['elec_imp','elec_exp'])]
msg.add_par('initial_activity_up',filtered_df) 

# msg.check_out()
df = msg.par('growth_activity_lo')
msg.remove_par('growth_activity_lo', df)
filtered_df = df[~df['technology'].isin(['elec_imp','elec_exp'])]
msg.add_par('growth_activity_lo',filtered_df) 

# msg.check_out()
df = msg.par('growth_activity_up')
msg.remove_par('growth_activity_up', df)
filtered_df = df[~df['technology'].isin(['elec_imp','elec_exp'])]
msg.add_par('growth_activity_up',filtered_df)

In [33]:
msg.commit('Remove bounds for import, export')

In [23]:
nodes = msg.set('node').tolist()

# Assuming you want to pass all nodes except 'World' and 'R11_GLB'
all_nodes = msg.set('node')
nodes_to_pass = [node for node in all_nodes if node not in ['World', 'R11_GLB']]

In [24]:
nodes_to_pass

['Canada',
 'Alberta',
 'BritishColumbia',
 'Manitoba',
 'NewBrunswick',
 'NewfoundlandandLabrador',
 'NorthwestTerritories',
 'NovaScotia',
 'Nunavut',
 'Ontario',
 'PrinceEdwardIsland',
 'Quebec',
 'Saskatchewan',
 'Yukon']

In [19]:

model_calibrate(msg, nodes)

>>>Warning<<<: No historical data for "['heat_i', 'heat_rc']" in "World"
>>>Warning<<<: No historical data for "['sp_el_I', 'sp_el_RC']" in "World"
>>>Warning<<<: No historical data for "['loil_trp', 'loil_i', 'loil_rc']" in "World"
>>>Warning<<<: No historical data for "['biomass_nc', 'biomass_rc', 'biomass_i']" in "World"
>>>Warning<<<: No historical data for "['biomass_nc', 'biomass_rc', 'biomass_i']" in "World"
>>>Warning<<<: No historical data for "['coal_rc', 'coal_i']" in "World"
>>>Warning<<<: No historical data for "['sp_eth_I', 'eth_i', 'eth_rc']" in "World"
>>>Warning<<<: No historical data for "['heat_i', 'heat_rc']" in "Canada"
>>>Warning<<<: No historical data for "['sp_el_I', 'sp_el_RC']" in "Canada"
>>>Warning<<<: No historical data for "['loil_trp', 'loil_i', 'loil_rc']" in "Canada"
>>>Warning<<<: No historical data for "['biomass_nc', 'biomass_rc', 'biomass_i']" in "Canada"
>>>Warning<<<: No historical data for "['biomass_nc', 'biomass_rc', 'biomass_i']" in "Canada"
>

/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:2204: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  act_ally.iloc[i]["value"] = max(act.iloc[0:i, :]["value"])
/Users/muhammadawais/Document

 There is no share for "biomass_rc" in relation "UE_res_comm_th_biomass" in node "Alberta" to be updated.
 There is no share for "biomass_rc" in relation "UE_res_comm_th_biomass" in node "BritishColumbia" to be updated.


/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)
/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)


 There is no share for "biomass_rc" in relation "UE_res_comm_th_biomass" in node "Manitoba" to be updated.
 There is no share for "biomass_rc" in relation "UE_res_comm_th_biomass" in node "NewBrunswick" to be updated.


/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)
/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)


... (19 outputs left out)

/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)


 Share of "elec_trp" in relation "UE_transport_electric_Minimum" in node "Quebec" was  updated from 0.0213662342375034 to 0.004
 Share of "elec_trp" in relation "UE_transport_electric_Minimum" in node "Saskatchewan" was  updated from 0.0213662342375034 to 0.0


/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)
/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/pre_process/utilities/message_utils.py:510: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  share = round(float(df_share["value"]) / df["value"].sum(), 3)


 Share of "elec_trp" in relation "UE_transport_electric_Minimum" in node "Yukon" was  updated from 0.0213662342375034 to 0.0


In [39]:
msg.commit('')

In [34]:
# Relations to be removed from copying
relation_remove = ["_min", "_max", "lim_", "_pot", "min-"]
cleanup_rel_tec(msg, nodes= nodes_to_pass, relation_remove=relation_remove)



KeyError: 'Canada'

In [51]:
# Correction of units
unit_correction(mp, msg)

 Notice: after this unit correction, you will have erorrs/issues if you use "old reporting" scripts!
- Unit was corrected for:
resource_remaining
resource_volume
technical_lifetime
capacity_factor
inv_cost
fix_cost
var_cost
abs_cost_activity_soft_up
abs_cost_activity_soft_lo
level_cost_activity_soft_up
level_cost_activity_soft_lo
soft_activity_up
soft_activity_lo
resource_cost
output
bound_activity_up
bound_activity_lo
initial_new_capacity_up
initial_new_capacity_lo
growth_new_capacity_up
growth_new_capacity_lo


In [35]:
remove_accounting = False

In [44]:
# Ensure DataFrame operations use 'reg' correctly
exclusion = []
relations = msg.set('relation').tolist()

if remove_accounting:
    rel_par = pd.DataFrame()
    for par in ['relation_upper', 'relation_lower', 'relation_cost']:
        # Ensure that DataFrame filtering here is correct
        df = msg.par(par)
        df = df[df['node_rel'].isin(reg)][['node_rel', 'relation']]
        rel_par = pd.concat([rel_par, df], ignore_index=True)

    rel_par = rel_par.drop_duplicates()

    rel_act = pd.DataFrame()
    for par in ['relation_new_capacity', 'relation_total_capacity', 'relation_activity']:
        df = msg.par(par)
        df = df[df['node_rel'].isin(reg)][['node_rel', 'relation']]
        rel_act = pd.concat([rel_act, df], ignore_index=True)

    rel_act = rel_act.drop_duplicates()

    exclusion = pd.concat([rel_act, rel_par]).drop_duplicates(keep=False)['relation'].tolist()
    exclusion += [r for r in relations if r not in rel_act['relation'].tolist() and r not in rel_par['relation'].tolist()]

# Add manual exclusions and process them
exclusion += ['PE_domestic_total', 'PE_import_total', 'PE_import_share', 'PE_export_total',
                'demb_limit', 'demF_limit', 'demI_limit', 'demp_limit', 'demR_limit', 'demt_limit',
                'elec_coal', 'elec_gas', 'elec_hydro', 'elec_nuclear', 'ele_nuc', 'foil_prod',
                'oil_based_elec_gen', 'gas_export', 'SO2_red_ref', 'SO2_red_synf', 'SO2_ind',
                'SO2_elec', 'SO2_import', 'SO2_r_c'] + [x for x in relation_remove if x in msg.set("relation")]

msg.check_out()
for rel in set(exclusion):
    if rel in msg.set('relation'):
        msg.remove_set('relation', rel)
        print('Relation: {} - deleted'.format(rel))

msg.commit('Remove unused elements')


## Solve Model

In [45]:
msg.model
msg.version
msg.commit('')

RuntimeError: this Scenario is not checked out, no changes to be committed!

In [66]:
caseName

'MESSAGEix_CA_provincial__baseline__v28'

In [46]:
# Exporting the built model (Scenario) to GAMS with an optional case name
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)

In [47]:
# Solving the model
msg.solve(solve_options={'barcrossalg': '2','lpmethod':'1'}, case=caseName)


--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 04/12/24 22:31:49 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix_CA_provincial__baseline__v31.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix_CA_provincial__baseline__v31.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix_CA_provincial__baseline__v31.gdx
Licensee: Small MUD - 5 User License         

2024-04-12 22:32:37,550 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2024-04-12 22:32:37,565 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [48]:

unit_correct = True  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060


In [85]:
#%% Generating reporter
rep = Reporter.from_scenario(msg, units={"apply": {"var_cost": "","inv_cost": "", "fix_cost": ""}})
report = rep.get('message::default') # in fact, only in = input x ACT, out = output x ACT is needed

if from_excel:
    # Read default report
    report_df = pd.read_csv('MESSAGE_NO_example_default_report.csv')
else:
# Or if trying another country model, execute the next 4:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

/Users/muhammadawais/miniconda3/envs/message3.8/lib/python3.10/site-packages/pyam/core.py:2691: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  index = pd.unique(index)
emission_factor: mixed units ['tC', '???', 'kg/kWa', '-'] discarded
ixmp.report.operator - WARNING: emission_factor: mixed units ['tC', '???', 'kg/kWa', '-'] discarded
/Users/muhammadawais/miniconda3/envs/message3.8/lib/python3.10/site-packages/pyam/core.py:2691: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  index = pd.unique(index)
/Users/muhammadawais/miniconda3/envs/message3.8/lib/python3.10/site-packages/pyam/core.py:2691: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  index = pd.unique(index)
/Users/muhammada

In [86]:
report_df

,Model,Scenario,Region,Variable,Unit,2010,2015,2020,2025,2030,2035,2040,2045,2050,2055,2060
0,MESSAGEix_CA_provincial,baseline,Alberta,CAP_NEW|new capacity|LNG_exp,,0.0,0.0,0.067645,0.109214,0.176326,0.28468,0.459617,0.8097,1.307267,2.110592,3.407566
1,MESSAGEix_CA_provincial,baseline,Alberta,CAP_NEW|new capacity|adipic_thermal,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
2,MESSAGEix_CA_provincial,baseline,Alberta,CAP_NEW|new capacity|ammonia_secloop,,NaN,NaN,NaN,NaN,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
3,MESSAGEix_CA_provincial,baseline,Alberta,CAP_NEW|new capacity|bio_hpl,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
4,MESSAGEix_CA_provincial,baseline,Alberta,CAP_NEW|new capacity|bio_hpl__air,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38150,MESSAGEix_CA_provincial,baseline,Yukon|Yukon,out|water_supply_constraint|water_constraint|e...,GWa,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
38151,MESSAGEix_CA_provincial,baseline,Yukon|Yukon,out|water_supply|freshwater_instream|extract__...,GWa,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
38152,MESSAGEix_CA_provincial,baseline,Yukon|Yukon,out|water_supply|freshwater_supply|extract__fr...,GWa,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
38153,MESSAGEix_CA_provincial,baseline,Yukon|Yukon,out|water_supply|saline_supply|extract__saline...,GWa,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000


In [87]:
# Plotted years
years = msg.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                           axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = pd.concat([out_flows , in_flows])

In [88]:
mp.close_db()

In [ ]:
report_df.to_csv('MESSAGE_CANADA_provincial_v4.csv')

In [94]:
report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|secondary|', 'Secondary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|secondary|', 'Secondary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('CAP_NEW|new capacity|', 'Capacity Addition|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('CAP|capacity|', 'Capacity|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('CAP|capacity|', 'Capacity|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('inv cost|', 'Capital Cost|', regex = False)

In [91]:
import pyam
import matplotlib.pyplot as plt
import pyam
plotyrs = [2020,2025,2030,2035,2040,2045,2050,2055,2060]
pyam_df = pyam.IamDataFrame(report_df)
pdf = pyam_df.copy()

/Users/muhammadawais/miniconda3/envs/message3.8/lib/python3.10/site-packages/pyam/utils.py:347: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.set_index(index + REQUIRED_COLS + extra_cols)
/Users/muhammadawais/miniconda3/envs/message3.8/lib/python3.10/site-packages/pyam/core.py:2691: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  index = pd.unique(index)


In [96]:
report_df.to_csv('MESSAGE_CANADA_provincial_baseline.csv')

In [93]:

biomass = ['Primary Energy|biomass|bio_istig_ccs|M1', 
'Primary Energy|biomass|bio_istig|M1']
'Primary Energy|biomass|bio_ppl|M1',
'Primary Energy|biomass|biomass_nc|M1',
'Primary Energy|biomass|biomass_t_d|M1',
'Primary Energy|biomass|eth_bio_ccs|M1',
'Primary Energy|biomass|eth_bio|M1',
'Primary Energy|biomass|gas_bio|M1',
'Primary Energy|biomass|h2_bio_ccs|M1',
'Primary Energy|biomass|h2_bio|M1',
'Primary Energy|biomass|liq_bio_ccs|M1',
'Primary Energy|biomass|liq_bio|M1','Primary Energy|biomass|land_use_biomass|M1']

lng = ['Primary Energy|LNG|LNG_bal|M1']

coal = ['Primary Energy|coal|coal_bal|M1',
                      'Primary Energy|coal|coal_exp|M1',
                      'Primary Energy|coal|coal_exp|M1',
                      'Primary Energy|coal|coal_extr_ch4|M1',
                      'Primary Energy|coal|coal_extr|M1',
                      'Primary Energy|coal|lignite_extr|M1']

gas = [#'Primary Energy|gas|gas_bal|M1',
    'Primary Energy|gas|gas_extr_1|M1',
      'Primary Energy|gas|gas_extr_2|M1',
      'Primary Energy|gas|gas_extr_3|M1',
      'Primary Energy|gas|gas_extr_4|M1',
      'Primary Energy|gas|gas_extr_5|M1',
      'Primary Energy|gas|gas_extr_6|M1']
      #'Primary Energy|gas|LNG_prod|M1']
      # 'Primary Energy|LNG|LNG_bal|M1']

oil = [#'Primary Energy|crudeoil|oil_bal|M1',
    'Primary Energy|crudeoil|oil_exp|M1',
     # 'Primary Energy|crudeoil|oil_extr_1_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_1|M1',
       # 'Primary Energy|crudeoil|oil_extr_2_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_2|M1',
      #  'Primary Energy|crudeoil|oil_extr_3_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_3|M1',
        #'Primary Energy|crudeoil|oil_extr_4_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_4|M1',
        'Primary Energy|crudeoil|oil_extr_5|M1',
        'Primary Energy|crudeoil|oil_extr_6|M1',
        'Primary Energy|crudeoil|oil_extr_7|M1']

hydro = ['Primary Energy|renewable|hydro|hydro_hc|M1',  'Primary Energy|renewable|hydro|hydro_lc|M1']

biomass_renew = ['Primary Energy|renewable|biomass|bio_extr_a|M1',
                'Primary Energy|renewable|biomass|bio_extr_b|M1',
                'Primary Energy|renewable|biomass|bio_extr_c|M1',
                'Primary Energy|renewable|biomass|bio_extr_d|M1',
                'Primary Energy|renewable|biomass|bio_extr_e|M1',
                'Primary Energy|renewable|biomass|bio_extr_f|M1',
                'Primary Energy|renewable|biomass|bio_extr_g|M1']

solar = ['Primary Energy|renewable|solar_pv|solar_pv_ppl|M1' , 
         'Primary Energy|renewable|solar_th|solar_th_ppl|M1']


wind = ['Primary Energy|renewable|wind|wind_ppf|M1', 'Primary Energy|renewable|wind|wind_ppl|M1']

lh2 = ['Primary Energy|lh2|lh2_bal|M1', 'Primary Energy|lh2|lh2_exp|M1']

ethanol = ['Primary Energy|methanol|meth_bal|M1',  'Primary Energy|methanol|meth_exp|M1']

methanol = ['Primary Energy|ethanol|eth_bal|M1', 'Primary Energy|ethanol|eth_exp|M1']



IndentationError: unindent does not match any outer indentation level (<tokenize>, line 24)

In [ ]:
pdf = pyam_df.copy()


In [ ]:
pdf.aggregate(
    "Primary Energy|coal",
    components=coal,
    append=True
)


pdf.aggregate(
    "Primary Energy|renewable|solar",
    components=solar,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|hydro",
    components=hydro,
    append=True
)

pdf.aggregate(
    "Primary Energy|ethanol",
    components=ethanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|methanol",
    components=methanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|Liq. Hydrogen",
    components=lh2,
    append=True
)
pdf.aggregate(
    "Primary Energy|renewable|wind",
    components=wind,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|biomass",
    components= biomass_renew,
    append=True
)


pdf.aggregate(
    "Primary Energy|biomass",
    components= biomass,
    append=True
)

pdf.aggregate(
    "Primary Energy|oil",
    components=oil,
    append=True
)

pdf.aggregate(
    "Primary Energy|gas",
    components=gas,
    append=True
)

pdf.aggregate(
    "Primary Energy|LNG",
    components=lng,
    append=True
)

In [ ]:
pdf.filter(variable = 'Primary Energy|gas').timeseries()

In [ ]:
data = pdf.filter( variable=["Primary Energy|renewable|solar",
                                            #"Primary Energy|LNG",
                                            "Primary Energy|gas",
                                            "Primary Energy|oil",
                                            "Primary Energy|biomass",
                                            "Primary Energy|renewable|wind",
                                            "Primary Energy|Liq. Hydrogen",
                                            "Primary Energy|methanol",
                                            "Primary Energy|ethanol",
                                            "Primary Energy|renewable|hydro",
                                            "Primary Energy|renewable|biomass",
                                            "Primary Energy|methanol",
                                           "Primary Energy|coal"],
              year = plotyrs, region = 'Yukon|Yukon')

data.plot.stack(title = 'Primary Energy by fuel - Baseline Scenario')
plt.legend(loc=1)
plt.tight_layout()
plt.show()

In [ ]:
i_feed = ['Useful Energy|i_feed|ethanol_fs|M1', 'Useful Energy|i_feed|gas_fs|M1',
         'Useful Energy|i_feed|loil_fs|M1', 'Useful Energy|i_feed|methanol_fs|M1']

transport = ['Useful Energy|transport|coal_trp|M1',
            'Useful Energy|transport|elec_trp|M1',
            'Useful Energy|transport|eth_fc_trp|M1',
            'Useful Energy|transport|eth_ic_trp|M1',
            'Useful Energy|transport|foil_trp|M1',
            'Useful Energy|transport|gas_trp|M1',
            'Useful Energy|transport|h2_fc_trp|M1',
            'Useful Energy|transport|loil_trp|M1',
            'Useful Energy|transport|meth_fc_trp|M1',
            'Useful Energy|transport|meth_ic_trp|M1']


i_spec = ['Useful Energy|i_spec|h2_fc_I|M1',
          'Useful Energy|i_spec|sp_coal_I|M1',
          'Useful Energy|i_spec|sp_el_I|M1',
          'Useful Energy|i_spec|sp_eth_I|M1',
          'Useful Energy|i_spec|sp_liq_I|M1',
          'Useful Energy|i_spec|sp_meth_I|M1']

i_therm =   ['Useful Energy|i_therm|biomass_i|M1',
            'Useful Energy|i_therm|coal_i|M1',
            'Useful Energy|i_therm|elec_i|M1',
            'Useful Energy|i_therm|eth_i|M1',
            'Useful Energy|i_therm|foil_i|M1',
            'Useful Energy|i_therm|gas_i|M1',
            'Useful Energy|i_therm|h2_fc_I|M1',
            'Useful Energy|i_therm|h2_i|M1',
            'Useful Energy|i_therm|heat_i|M1',
            'Useful Energy|i_therm|hp_el_i|M1',
            'Useful Energy|i_therm|hp_gas_i|M1',
            'Useful Energy|i_therm|loil_i|M1',
            'Useful Energy|i_therm|meth_i|M1',
            'Useful Energy|i_therm|solar_i|M1']
rc_spec = ['Useful Energy|rc_spec|h2_fc_RC|M1',
          'Useful Energy|rc_spec|sp_el_RC|M1']

rc_therm = ['Useful Energy|rc_therm|biomass_rc|M1',
            'Useful Energy|rc_therm|coal_rc|M1',
            'Useful Energy|rc_therm|eth_rc|M1',
            'Useful Energy|rc_therm|foil_rc|M1',
            'Useful Energy|rc_therm|gas_rc|M1',
            'Useful Energy|rc_therm|h2_fc_RC|M1',
            'Useful Energy|rc_therm|h2_rc|M1',
            'Useful Energy|rc_therm|heat_rc|M1',
            'Useful Energy|rc_therm|hp_el_rc|M1',
            'Useful Energy|rc_therm|hp_gas_rc|M1',
            'Useful Energy|rc_therm|loil_rc|M1',
            'Useful Energy|rc_therm|meth_rc|M1',
            'Useful Energy|rc_therm|solar_rc|M1']

biom = ['Useful Energy|non-comm|biomass_nc|M1']

In [ ]:
df_useful = pyam_df.copy()
# df_useful.aggregate(
#     "Useful Energy|non-Commercial biomass",
#     components=biom,
#     append=True
# )

df_useful.aggregate(
    "Useful Energy|residential/commercial thermal",
    components=rc_therm,
    append=True
)

df_useful.aggregate(
    "Useful Energy|residential/commercial specific",
    components=rc_spec,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial thermal",
    components=i_therm,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial specific",
    components=i_spec,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial feedstock",
    components=i_feed,
    append=True
)

df_useful.aggregate(
    "Useful Energy|transport",
    components=i_therm,
    append=True
)


In [ ]:
data = df_useful.filter(scenario = 'baseline', variable=[
    "Useful Energy|non-Commercial biomass",
    "Useful Energy|residential/commercial thermal",
    "Useful Energy|residential/commercial specific",
    "Useful Energy|industrial thermal",
    "Useful Energy|industrial specific",
    "Useful Energy|industrial feedstock",
    "Useful Energy|transport"],
              year = plotyrs, region = 'Yukon|Yukon')


data.plot.stack(title = 'Demands by sector')
plt.legend(loc=1)
plt.tight_layout()
plt.show()

In [ ]:
cement = ['Emissions|CO2|cement_CO2|M1']
coal = ['Emissions|CO2|coal_extr_ch4|M1',
        'Emissions|CO2|coal_extr|M1',
        'Emissions|CO2|coal_imp|M1']
foil = ['Emissions|CO2|foil_imp|M1']
gas = ['Emissions|CO2|gas_extr_1|M1',
        'Emissions|CO2|gas_extr_2|M1',
        'Emissions|CO2|gas_extr_3|M1',
        'Emissions|CO2|gas_extr_4|M1',
        'Emissions|CO2|gas_extr_5|M1',
        'Emissions|CO2|gas_extr_6|M1'
        ]
lignite = ['Emissions|CO2|lignite_extr|M1']
loil = ['Emissions|CO2|loil_imp|M1']
meth_ccs = ['Emissions|CO2|meth_coal_ccs|M1']
oil = ['Emissions|CO2|oil_extr_1_ch4|M1',
    'Emissions|CO2|oil_extr_1|M1',
    'Emissions|CO2|oil_extr_2_ch4|M1',
    'Emissions|CO2|oil_extr_2|M1',
    'Emissions|CO2|oil_extr_3_ch4|M1',
    'Emissions|CO2|oil_extr_3|M1',
    'Emissions|CO2|oil_extr_4_ch4|M1',
    'Emissions|CO2|oil_extr_4|M1',
    'Emissions|CO2|oil_extr_5|M1',
    'Emissions|CO2|oil_extr_6|M1',
    'Emissions|CO2|oil_extr_7|M1']

oil_trade = ['Emissions|CO2|foil_imp|M1',
             'Emissions|CO2|loil_exp|M1',
            'Emissions|CO2|loil_imp|M1',
             'Emissions|CO2|oil_imp|M1']
       
gas_trade = ['Emissions|CO2|LNG_exp|M1',
            'Emissions|CO2|LNG_imp|M1',
            'Emissions|CO2|gas_imp|M1']
            #'Emissions|CO2|LNG_exp|M1']

In [ ]:
df_emis_b = pyam_df.filter(variable = 'Emissions|CO2|*',year = plotyrs)

In [ ]:
df_emis_b.to_csv('emissions.csv')

In [ ]:
df_emis = pyam_df.copy()


df_emis.aggregate(
    "Emissions|CO2|gas",
    components=gas,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil",
    components=oil,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|coal",
    components=coal,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|cement",
    components=cement,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil trade",
    components=oil_trade,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|gas trade",
    components=gas_trade,
    append=True
)

In [ ]:
data = df_emis.filter(scenario = 'baseline', variable=
    ["Emissions|CO2|gas",
    "Emissions|CO2|oil",
    "Emissions|CO2|coal",
    "Emissions|CO2|cement",
    "Emissions|CO2|oil trade",
    "Emissions|CO2|gas trade"],
              year = plotyrs, region = 'Alberta')

data.plot.stack(title = 'CO2 Emissions - Baseline Scenario')
plt.legend(loc=1)
plt.tight_layout()
plt.show()

In [ ]:
df_emis.aggregate('Emissions|*').timeseries()

In [ ]:

coal = ['Total Costs|coal_adv',
        'Total Costs|coal_adv_ccs',
        'Total Costs|coal_bal',
        'Total Costs|coal_exp',
        'Total Costs|coal_extr',
        'Total Costs|coal_extr_ch4',
        'Total Costs|coal_gas',
        'Total Costs|coal_hpl',
        'Total Costs|coal_i',
        'Total Costs|coal_imp',
        'Total Costs|coal_ppl',
        'Total Costs|coal_ppl_u',
        'Total Costs|coal_rc',
        'Total Costs|coal_ppl',
        'Total Costs|coal_ppl_u',
       'Total Costs|coal_t_d',
       'Total Costs|syn_liq',
        'Total Costs|meth_coal',
       'Total Costs|lignite_extr']
biomass = ['bio_hpl', 'bio_ppl',
          'Total Costs|liq_bio']


elec = ['Total Costs|elec_exp',
        'Total Costs|elec_i',
        'Total Costs|elec_imp',
        'Total Costs|elec_t_d',
        'Total Costs|elec_trp']

oil = ['Total Costs|foil_i',
    'Total Costs|foil_imp',
    'Total Costs|foil_ppl',
    'Total Costs|oil_extr_1'
    'Total Costs|oil_extr_1_ch4'
    'Total Costs|oil_extr_2',
    'Total Costs|oil_extr_3',
    'Total Costs|oil_extr_4',
    'Total Costs|oil_extr_5',
    'Total Costs|oil_extr_6',
    'Total Costs|oil_extr_7',
    'Total Costs|oil_extr_2_ch4',
    'Total Costs|oil_extr_3_ch4',
    'Total Costs|loil_cc',
    'Total Costs|loil_ppl']
gas = ['Total Costs|gas_bal',
    'Total Costs|gas_bio',
    'Total Costs|gas_cc',
    'Total Costs|gas_cc_ccs',
    'Total Costs|gas_ct',
    'Total Costs|gas_extr_1',
    'Total Costs|gas_extr_2',
    'Total Costs|gas_extr_3',
    'Total Costs|gas_extr_4',
    'Total Costs|gas_extr_5',
    'Total Costs|gas_extr_6',
    'Total Costs|gas_extr_mpen',
    'Total Costs|gas_fs',
    'Total Costs|gas_hpl',
    'Total Costs|gas_i',
    'Total Costs|gas_imp',
    'Total Costs|gas_ppl',
    'Total Costs|gas_rc',
    'Total Costs|gas_t_d',
    'Total Costs|gas_t_d_ch4',
    'Total Costs|gas_trp',
      'Total Costs|igcc']

geo_ppl = ['Total Costs|geo_hpl','Total Costs|geo_ppl']

solar = ['Total Costs|solar_i',
        'Total Costs|solar_pv_ppl',
        'Total Costs|solar_rc',
        'Total Costs|solar_th_ppl']

wind = ['Total Costs|wind_curtailment1',
'Total Costs|wind_curtailment2',
'Total Costs|wind_curtailment3',
'Total Costs|wind_ppf',
'Total Costs|wind_ppl']

hydro = ['Total Costs|hydro_hc',
'Total Costs|hydro_lc']

nuclear = ['Total Costs|nuc_lc']


In [ ]:
df_cost = pyam_df.copy()


df_cost.aggregate(
    "Total Costs|coal",
    components=coal,
    append=True
)

df_cost.aggregate(
    "Total Costs|biomass",
    components=biomass,
    append=True
)

df_cost.aggregate(
    "Total Costs|electricity",
    components=elec,
    append=True
)

df_cost.aggregate(
    "Total Costs|oil",
    components=biomass,
    append=True
)

df_cost.aggregate(
    "Total Costs|gas",
    components=gas,
    append=True
)

df_cost.aggregate(
    "Total Costs|geothermal",
    components=geo_ppl,
    append=True
)

df_cost.aggregate(
    "Total Costs|solar",
    components=solar,
    append=True
)

df_cost.aggregate(
    "Total Costs|wind",
    components=wind,
    append=True
)

df_cost.aggregate(
    "Total Costs|hydro",
    components=hydro,
    append=True
)
df_cost.aggregate(
    "Total Costs|nuclear",
    components=nuclear,
    append=True
)

In [ ]:
data = df_cost.filter(scenario = 'baseline', variable=
    ["Total Costs|coal","Total Costs|biomass",
    "Total Costs|electricity",
    "Total Costs|oil",
    "Total Costs|gas",
    "Total Costs|geothermal",
    "Total Costs|solar",
    "Total Costs|wind",
    "Total Costs|hydro",
    "Total Costs|nuclear"],
              year = plotyrs, region = 'Alberta')

data.plot.stack(title = 'Total (Operational+ Investment) Costs - Baseline')
plt.legend(loc=1)
plt.tight_layout()
plt.show()


Now making another scenario cloned from baseline to add other policies. 

Policy No. 1
Baseline - NDC 
Implement Canada's commitments 


In [ ]:
msgbas = msg.clone(modelName, 'Emissons bounds','Introducing Canadas goals',
                     keep_solution=False)

In [ ]:
msgbas.check_out()

In [ ]:
msg.par('emiss')

In [ ]:

def expand_grid(dictionary):
    return pd.DataFrame([row for row in product(*dictionary.values())], 
                       columns=dictionary.keys())


In [ ]:
  
    
year_df = [2020, 2025, 2030, 2035, 2040 ,2045 ,2050 ,2055 ,2060 ]


dic = {
    'node': [nodeName],
    'type_emission':[str('TCE')],
    'type_tec':[str('all')],
    'type_year': [2020, 2025, 2030, 2035, 2040 ,2045 ,2050 ,2055 ,2060 ],
    'value':     [629, 572, 526, 450 , 375, 320, 255, 200, 150],
    'unit':[str('-')]}

df = expand_grid(dic)
df = df.iloc[0:9]
df['type_year'] = year_df



In [ ]:
# msgbas.remove_solution()

In [ ]:
msgbas.check_out()

In [ ]:
#msgSDG.add_par('bound_emission',df)
msgbas.add_par('bound_emission', df)
msgbas.commit('emissions bounded ')

In [ ]:
msgbas.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

In [ ]:
from message_ix.reporting import Reporter
from update_units import unit_correction
import numpy as np
unit_correct = False  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060

if unit_correct:
    msg = msg.clone(keep_solution=False)
    unit_correction(mp, msg)
    caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)
    # Solving the model
    msg.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

#%% Generating reporter
rep = Reporter.from_scenario(msgbas)
report = rep.get('message:default') # in fact, only in = input x ACT, out = output x ACT is needed

if from_excel:
    # Read default report
    report_df = pd.read_csv('MESSAGE_NO_example_default_report.csv')
else:
# Or if trying another country model, execute the next 4:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

    # Plotted years
years = msgbas.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                           axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = out_flows.append(in_flows)

In [ ]:
report_df.to_csv('MESSAGE_CANADA_baselinendc.csv')

In [ ]:
report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('emis', 'Emissions', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)

In [ ]:
import pyam
import matplotlib.pyplot as plt
import pyam
plotyrs = [2020,2025,2030,2035,2040,2045,2050,2055,2060]
pyam_df = pyam.IamDataFrame(report_df)
pdf = pyam_df.copy()

In [ ]:
pdf.aggregate(
    "Primary Energy|coal",
    components=coal,
    append=True
)


pdf.aggregate(
    "Primary Energy|renewable|solar",
    components=solar,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|hydro",
    components=hydro,
    append=True
)

pdf.aggregate(
    "Primary Energy|ethanol",
    components=ethanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|methanol",
    components=methanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|Liq. Hydrogen",
    components=lh2,
    append=True
)
pdf.aggregate(
    "Primary Energy|renewable|wind",
    components=wind,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|biomass",
    components= biomass_renew,
    append=True
)


pdf.aggregate(
    "Primary Energy|biomass",
    components= biomass,
    append=True
)

pdf.aggregate(
    "Primary Energy|oil",
    components=oil,
    append=True
)

pdf.aggregate(
    "Primary Energy|gas",
    components=gas,
    append=True
)

pdf.aggregate(
    "Primary Energy|LNG",
    components=lng,
    append=True
)

In [ ]:
pdf.filter(variable = 'Primary Energy|oil').timeseries()

In [ ]:
pdf.filter( variable=["Primary Energy|renewable|solar",
                                            #"Primary Energy|LNG",
                                            "Primary Energy|gas",
                                            "Primary Energy|oil",
                                            "Primary Energy|biomass",
                                            "Primary Energy|renewable|wind",
                                            "Primary Energy|Liq. Hydrogen",
                                            "Primary Energy|methanol",
                                            "Primary Energy|ethanol",
                                            "Primary Energy|renewable|hydro",
                                            "Primary Energy|renewable|biomass",
                                            "Primary Energy|methanol",
                                           "Primary Energy|coal"],
              year = plotyrs).stack_plot(total=True,
                                        title = 'Primary Energy by fuel - Baseline Scenario')

In [ ]:
df_emis = pyam_df.copy()


df_emis.aggregate(
    "Emissions|CO2|gas",
    components=gas,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil",
    components=oil,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|coal",
    components=coal,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|cement",
    components=cement,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil trade",
    components=oil_trade,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|gas trade",
    components=gas_trade,
    append=True
)

In [ ]:
df_emis.timeseries()

In [ ]:
df_emis.filter( variable=
    ["Emissions|CO2|gas",
    "Emissions|CO2|oil",
    "Emissions|CO2|coal",
    "Emissions|CO2|cement",
    "Emissions|CO2|oil trade",
    "Emissions|CO2|gas trade"],
              year = plotyrs,region = 'Canada').stack_plot(total=True,
                                         title = 'CO2 Emissions - Baseline Scenario')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('MESSAGE_CANADA_baselinendc.csv')

In [ ]:
df['Model'].replace('CD_Links_SSP2_R14', 'MESSAGEix-Canada', inplace=True)

In [ ]:
df.head()

In [ ]:
df.to_csv('MESSAGE_CANADA_v1.csv')

In [ ]:
import pandas as pd

In [ ]:
iso = 'CAN'

In [ ]:
# SSP GDP data
ssp_gdp_df = pd.read_excel( str('OECD_SSP_GDP_PPP.xlsx'),
        'OECD_SSP_GDP_PPP',
        header = 0)
ssp_gdp_df = ssp_gdp_df.loc[ (  ssp_gdp_df.Region == iso ) ].reset_index(drop=True)

# SSP population data

ssp_pop_df = pd.read_excel(str('OECD_SSP_POP.xlsx'),
        'OECD_SSP_POP',
        header = 0)
ssp_pop_df = ssp_pop_df.loc[ (  ssp_pop_df.Region == iso  ) ].reset_index(drop=True)

In [ ]:
ssp_pop_df['Scenario'] = ssp_pop_df['Scenario'].str.replace('_v9_130325', '')
ssp_gdp_df['Scenario'] = ssp_gdp_df['Scenario'].str.replace('_v9_130325', '')

In [ ]:
ssp_gdp_df

In [ ]:
import pyam
import matplotlib.pyplot as plt
import pyam

pop_df = pyam.IamDataFrame(ssp_pop_df)
gdp_df = pyam.IamDataFrame(ssp_gdp_df)

In [ ]:
pop_df.head()

In [ ]:
pop_df.plot(color='scenario', title='Population Projections of Canada across SSPs')

In [ ]:
gdp_df.plot(color='scenario', title='GDP Projections of Canada across SSPs')

In [ ]:
mp.add_unit('tCO2/kWa')
mp.add_unit('MtCO2')

In [ ]:
msgbas.set('type_emission')

In [ ]:

msgbas.add_par('bound_emission', ['Canada', 'TCE', 'all', 'cumulative'],
             value=700., unit='MtCO2')

In [ ]:
msgbas.commit('adding emissions bound')

In [ ]:
msgbas.solve()

In [ ]:
msgbas.scenario

In [ ]:
msgbas.par('bound_emission')

In [ ]:
msgCT = msgbas.clone(modelName, 'carbon_bound_by_year','introducing a carbon tax',
                             keep_solution=False)
msgCT.check_out()

In [ ]:
emissions = msgCT.var('EMISS')
emissions

In [ ]:

emission_prices_by_year = msg.var('PRICE_EMISSION')
emission_prices_by_year

In [ ]:

scen_tax = base.clone(model, 'carbon_tax','introducing a carbon tax',
                      keep_solution=False)
scen_tax.check_out()

In [ ]:
years

In [ ]:

base_tax_emission = {
    'node': 'Canada',
    'type_year': years,
    'type_tec': 'all',
    'unit': 'USD/tCO2',
    'type_emission': 'GHG',
    'value': emission_prices.set_index('year').lvl
}

In [ ]:


# in case you need to add the new unit to the platform
mp.add_unit('USD/tCO2')

tax_emission = make_df(base_tax_emission)
scen_tax.add_par('tax_emission', tax_emission)
scen_tax.par('tax_emission')

In [ ]:

scen_tax.commit(comment='setting taxes on emissions')

In [ ]:
scen_tax.var('EMISS', {'node': 'Westeros'})